# Simulation study evaluation plots

Reads the tidy outputs of `extract_trans_eval.R` (`trans_eval_long.csv`) and
`extract_cis_mse.py` (`cis_eval.csv`) -- see `docs/SIMULATION_STUDY_PLAN.md` §8.

**Assumptions made where the plot spec was ambiguous** (isolated to the helper
functions below -- check against real output and adjust if wrong):
- TPR/prop-additive-hill plots: "facets for the different y_ntc values" + "additionally
  facetted by X/Y" is implemented as nested `facet_grid` rows (e.g.
  `cells_per_gene + y_ntc_log2 ~ n_guides`), not a separate figure per y_ntc.
- TPR's continuous x-axis (`prop_log2FC_observed`) is binned (8 equal-width bins, 0-1)
  to compute a rate per bin; `full_log2FC_true` is used directly as discrete color
  (only 4 grid values, no binning needed).
- "Prop. additive_hill fit" is built with both facet schemes, mirroring the paired
  FPR/TPR plots.
- **3a (n) and 3d (full log2FC)** are boxplots: x = binned `prop_log2FC_observed`,
  fill = the true value (discrete grid), box = distribution of the estimate, black "x"
  markers = the true value at the matching dodge position. 3d is restricted to
  single_hill-truth features only (a binned-proportion x-axis isn't meaningful for
  null-truth rows, which have no `prop_log2FC_observed`) -- this drops null features
  from 3d, unlike the original scatter version.
- **3c (y_ntc)** is a boxplot: x = true `y_ntc_log2` (already a 4-value discrete grid),
  fill = true `o_y_log2` (already discrete, 2 values), same "x" true-value marker
  convention. Applies to all features (both null and single_hill truth).
- **3b (EC50)** stays a scatter plot, now produced with **two fill variants** per facet
  scheme: the original `prop_log2FC_observed`, and an alternate `guide_log2_range`
  (`log2(x_max) - log2(x_min)` of the guides' achieved cis-expression range for that
  scenario/replicate -- a scenario-level quantity, constant across features within one
  scenario/replicate).
- **FPR bars (1a/1b)** now have the group's `n` (feature count) printed above each bar,
  to distinguish "genuinely FPR=0" from "no data for this grid cell" -- both look like
  an empty panel otherwise.


In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
})

In [ ]:
# --- Config -- edit these paths ---
trans_eval_path <- "trans_eval_long.csv"
cis_eval_path <- "cis_eval.csv"
outdir <- "./eval_plots"
dir.create(outdir, showWarnings = FALSE, recursive = TRUE)

In [ ]:
save_plot <- function(p, name, width = 10, height = 8) {
  # Displays inline (IRkernel auto-prints the last expression of a cell, but not
  # inside a loop/function call, so this prints explicitly) and saves to PDF.
  print(p)
  path <- file.path(outdir, paste0(name, ".pdf"))
  ggsave(path, p, width = width, height = height, limitsize = FALSE)
  invisible(p)
}

In [ ]:
trans_eval <- fread(trans_eval_path)
cis_eval <- fread(cis_eval_path)

# Shared derived columns
trans_eval[, y_ntc_log2_fit := log2(y_ntc)]
trans_eval[, prop_bin := cut(prop_log2FC_observed, breaks = seq(0, 1, by = 0.125),
                              include.lowest = TRUE)]
cat(sprintf("Loaded %d trans_eval rows, %d cis_eval rows\n", nrow(trans_eval), nrow(cis_eval)))

## 1. False positive rate

FPR per `(y_ntc_log2, o_y_log2, <facet vars>)` = fraction of `no_effect`-truth features called positive, pooling across matching scenarios/replicates. Bar labels show `n` (feature count in that group) -- a bar that's absent (not just short) means zero features fell in that grid cell, not FPR=0.

In [ ]:
fpr_data <- trans_eval[effect_type == "no_effect"]

make_fpr_plot <- function(dt, facet_rows, facet_cols, title_suffix) {
  agg <- dt[, .(FPR = mean(is_positive, na.rm = TRUE), n = .N),
            by = c("y_ntc_log2", "o_y_log2", facet_rows, facet_cols)]
  ggplot(agg, aes(x = factor(y_ntc_log2), y = FPR, fill = factor(o_y_log2))) +
    geom_col(position = position_dodge(width = 0.8), width = 0.7) +
    geom_text(aes(label = n), position = position_dodge(width = 0.8),
              vjust = -0.3, size = 2.5) +
    geom_hline(yintercept = 0.05, linetype = "dashed", color = "red") +
    facet_grid(reformulate(facet_cols, paste(facet_rows, collapse = "+"))) +
    labs(x = "y_ntc (log2)", y = "False positive rate", fill = "y overdispersion (log2)",
         title = paste("FPR (bar labels = n features) --", title_suffix)) +
    theme_bw()
}

In [ ]:
save_plot(make_fpr_plot(fpr_data, "cells_per_gene", "n_guides",
                         "faceted by n_guides (x) x cells_per_gene (y)"),
          "1a_fpr_by_guides_cells")

In [ ]:
save_plot(make_fpr_plot(fpr_data, "log2_X_NTC", "log2_o_x",
                         "faceted by cis o_x (x) x cis log2_X_NTC (y)"),
          "1b_fpr_by_cis_ox_xntc")

## 2. True positive rate (+ prop. additive_hill fit)

Restricted to single_hill-truth features whose fit came out `single_hill` or `not_dependent` ("flat") -- excludes `additive_hill` fits, reported separately below as their own proportion.

In [ ]:
tpr_base <- trans_eval[effect_type == "single_hill" &
                          fit_type %in% c("single_hill", "not_dependent")]

make_tpr_plot <- function(dt, facet_rows, facet_cols, title_suffix) {
  agg <- dt[!is.na(prop_bin), .(TPR = mean(is_positive, na.rm = TRUE), n = .N),
            by = c("prop_bin", "full_log2FC_true", "y_ntc_log2", facet_rows, facet_cols)]
  ggplot(agg, aes(x = prop_bin, y = TPR, color = factor(full_log2FC_true))) +
    geom_point(aes(size = n), alpha = 0.7) +
    geom_line(aes(group = factor(full_log2FC_true))) +
    facet_grid(reformulate(facet_cols, paste(c(facet_rows, "y_ntc_log2"), collapse = "+"))) +
    labs(x = "Proportion of full log2FC observed (binned)", y = "True positive rate",
         color = "True full log2FC", size = "n features",
         title = paste("TPR (single_hill/flat fits only) --", title_suffix)) +
    theme_bw() + theme(axis.text.x = element_text(angle = 45, hjust = 1))
}

In [ ]:
save_plot(make_tpr_plot(tpr_base, "cells_per_gene", "n_guides",
                         "faceted by n_guides (x) x cells_per_gene+y_ntc (y)"),
          "2a_tpr_by_guides_cells", height = 14)

In [ ]:
save_plot(make_tpr_plot(tpr_base, "log2_X_NTC", "log2_o_x",
                         "faceted by cis o_x (x) x cis log2_X_NTC+y_ntc (y)"),
          "2b_tpr_by_cis_ox_xntc", height = 14)

In [ ]:
prop_add_base <- trans_eval[effect_type == "single_hill"]
prop_add_base[, is_additive := fit_type == "additive_hill"]

make_prop_additive_plot <- function(dt, facet_rows, facet_cols, title_suffix) {
  agg <- dt[!is.na(prop_bin), .(prop_additive = mean(is_additive, na.rm = TRUE), n = .N),
            by = c("prop_bin", "full_log2FC_true", "y_ntc_log2", facet_rows, facet_cols)]
  ggplot(agg, aes(x = prop_bin, y = prop_additive, color = factor(full_log2FC_true))) +
    geom_point(aes(size = n), alpha = 0.7) +
    geom_line(aes(group = factor(full_log2FC_true))) +
    facet_grid(reformulate(facet_cols, paste(c(facet_rows, "y_ntc_log2"), collapse = "+"))) +
    labs(x = "Proportion of full log2FC observed (binned)", y = "Proportion fit as additive_hill",
         color = "True full log2FC", size = "n features",
         title = paste("Prop. additive_hill fit (single_hill truth) --", title_suffix)) +
    theme_bw() + theme(axis.text.x = element_text(angle = 45, hjust = 1))
}

In [ ]:
save_plot(make_prop_additive_plot(prop_add_base, "cells_per_gene", "n_guides",
                                   "faceted by n_guides (x) x cells_per_gene+y_ntc (y)"),
          "2c_prop_additive_by_guides_cells", height = 14)

In [ ]:
save_plot(make_prop_additive_plot(prop_add_base, "log2_X_NTC", "log2_o_x",
                                   "faceted by cis o_x (x) x cis log2_X_NTC+y_ntc (y)"),
          "2d_prop_additive_by_cis_ox_xntc", height = 14)

## 3. Estimated vs true

### 3a. Hill exponent `n`

Boxplot of the estimated `n` (active-component match) within each binned `prop_log2FC_observed` x true-n group; black × marks the true value at the matching dodge position. Restricted to single_hill-truth features with a resolved active-component estimate.

In [ ]:
sh_active_n <- trans_eval[effect_type == "single_hill" & !is.na(estimated_n) & !is.na(prop_bin)]

make_n_boxplot <- function(dt, facet_rows, facet_cols, title_suffix) {
  ggplot(dt, aes(x = prop_bin, y = estimated_n, fill = factor(n_true))) +
    geom_boxplot(outlier.size = 0.5, position = position_dodge(width = 0.8)) +
    geom_point(aes(y = n_true), shape = 4, position = position_dodge(width = 0.8),
               color = "black", size = 1.5) +
    facet_grid(reformulate(facet_cols, facet_rows)) +
    labs(x = "Prop. full log2FC observed (binned)", y = "Estimated n", fill = "True n",
         title = paste("n estimated vs true --", title_suffix)) +
    theme_bw() + theme(axis.text.x = element_text(angle = 45, hjust = 1))
}

In [ ]:
save_plot(make_n_boxplot(sh_active_n, "cells_per_gene", "n_guides",
                          "faceted by n_guides (x) x cells_per_gene (y)"),
          "3a_n_est_vs_true_guides_cells")

In [ ]:
save_plot(make_n_boxplot(sh_active_n, "log2_X_NTC", "log2_o_x",
                          "faceted by cis o_x (x) x cis log2_X_NTC (y)"),
          "3a_n_est_vs_true_cis_ox_xntc")

### 3b. EC50 (log2FC)

Scatter plot, x = true EC50 (log2FC), y = estimated EC50 (log2FC). Two fill variants: the original `prop_log2FC_observed`, and an alternate `guide_log2_range` (scenario-level guide dynamic range).

In [ ]:
sh_active_k <- trans_eval[effect_type == "single_hill" & !is.na(estimated_K_log2FC)]

make_ec50_plot <- function(dt, facet_rows, facet_cols, fill_col, fill_lab, title_suffix) {
  ggplot(dt, aes(x = K_log2FC_true, y = estimated_K_log2FC, color = .data[[fill_col]])) +
    geom_point(alpha = 0.5, size = 0.8) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey40") +
    facet_grid(reformulate(facet_cols, facet_rows)) +
    scale_color_viridis_c(na.value = "grey70") +
    labs(x = "True EC50 (log2FC)", y = "Estimated EC50 (log2FC)", color = fill_lab,
         title = paste("EC50 estimated vs true --", title_suffix)) +
    theme_bw()
}

In [ ]:
save_plot(make_ec50_plot(sh_active_k, "cells_per_gene", "n_guides",
                          "prop_log2FC_observed", "Prop. log2FC observed",
                          "faceted by n_guides (x) x cells_per_gene (y)"),
          "3b_EC50_est_vs_true_guides_cells_propobs")

In [ ]:
save_plot(make_ec50_plot(sh_active_k, "log2_X_NTC", "log2_o_x",
                          "prop_log2FC_observed", "Prop. log2FC observed",
                          "faceted by cis o_x (x) x cis log2_X_NTC (y)"),
          "3b_EC50_est_vs_true_cis_ox_xntc_propobs")

In [ ]:
save_plot(make_ec50_plot(sh_active_k, "cells_per_gene", "n_guides",
                          "guide_log2_range", "Guide log2 range",
                          "faceted by n_guides (x) x cells_per_gene (y)"),
          "3b_EC50_est_vs_true_guides_cells_guiderange")

In [ ]:
save_plot(make_ec50_plot(sh_active_k, "log2_X_NTC", "log2_o_x",
                          "guide_log2_range", "Guide log2 range",
                          "faceted by cis o_x (x) x cis log2_X_NTC (y)"),
          "3b_EC50_est_vs_true_cis_ox_xntc_guiderange")

### 3c. `y_ntc`

Boxplot, since both x (true `y_ntc_log2`) and fill (true `o_y_log2`) are already discrete grids. Black × marks the true value. Applies to all features (null and single_hill truth).

In [ ]:
make_yntc_boxplot <- function(dt, facet_rows, facet_cols, title_suffix) {
  ggplot(dt[!is.na(y_ntc_log2_fit)], aes(x = factor(y_ntc_log2), y = y_ntc_log2_fit,
                                          fill = factor(o_y_log2))) +
    geom_boxplot(outlier.size = 0.5, position = position_dodge(width = 0.8)) +
    geom_point(aes(y = y_ntc_log2), shape = 4, position = position_dodge(width = 0.8),
               color = "black", size = 1.5) +
    facet_grid(reformulate(facet_cols, facet_rows)) +
    labs(x = "True y_ntc (log2)", y = "Estimated y_ntc (log2)",
         fill = "True y overdispersion (log2)",
         title = paste("y_ntc estimated vs true --", title_suffix)) +
    theme_bw()
}

In [ ]:
save_plot(make_yntc_boxplot(trans_eval, "cells_per_gene", "n_guides",
                             "faceted by n_guides (x) x cells_per_gene (y)"),
          "3c_yntc_est_vs_true_guides_cells")

In [ ]:
save_plot(make_yntc_boxplot(trans_eval, "log2_X_NTC", "log2_o_x",
                             "faceted by cis o_x (x) x cis log2_X_NTC (y)"),
          "3c_yntc_est_vs_true_cis_ox_xntc")

### 3d. Full log2FC

Boxplot, same binned-`prop_log2FC_observed` x-axis and true-value marker convention as 3a. Restricted to single_hill-truth features (a binned-proportion x-axis isn't meaningful for null-truth rows).

In [ ]:
sh_fc <- trans_eval[effect_type == "single_hill" & !is.na(prop_bin) & !is.na(full_log2fc_median)]

make_fulllog2fc_boxplot <- function(dt, facet_rows, facet_cols, title_suffix) {
  ggplot(dt, aes(x = prop_bin, y = full_log2fc_median, fill = factor(full_log2FC_true))) +
    geom_boxplot(outlier.size = 0.5, position = position_dodge(width = 0.8)) +
    geom_point(aes(y = full_log2FC_true), shape = 4, position = position_dodge(width = 0.8),
               color = "black", size = 1.5) +
    facet_grid(reformulate(facet_cols, facet_rows)) +
    labs(x = "Prop. full log2FC observed (binned)", y = "Estimated full log2FC",
         fill = "True full log2FC",
         title = paste("full log2FC estimated vs true --", title_suffix)) +
    theme_bw() + theme(axis.text.x = element_text(angle = 45, hjust = 1))
}

In [ ]:
save_plot(make_fulllog2fc_boxplot(sh_fc, "cells_per_gene", "n_guides",
                                   "faceted by n_guides (x) x cells_per_gene (y)"),
          "3d_fulllog2fc_est_vs_true_guides_cells")

In [ ]:
save_plot(make_fulllog2fc_boxplot(sh_fc, "log2_X_NTC", "log2_o_x",
                                   "faceted by cis o_x (x) x cis log2_X_NTC (y)"),
          "3d_fulllog2fc_est_vs_true_cis_ox_xntc")

## 4. fit_cis accuracy: MSE of log2(x_true)

One figure per `guide_shape`. Note the grid has 4 `log2_X_NTC` levels (-1/0/1/2), giving 4 facet rows, not 3.

In [ ]:
cis_plot_data <- cis_eval[!is.na(mse_log2_x_true)]

for (shape in unique(cis_plot_data$guide_shape)) {
  d <- cis_plot_data[guide_shape == shape]
  p <- ggplot(d, aes(x = factor(cells_per_gene), y = mse_log2_x_true, fill = factor(n_guides))) +
    geom_point(position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.7),
               shape = 21, size = 2, alpha = 0.7) +
    stat_summary(aes(group = factor(n_guides)), fun = mean, geom = "line",
                 position = position_dodge(width = 0.7), color = "black") +
    facet_grid(log2_X_NTC ~ log2_o_x, labeller = label_both) +
    labs(x = "Cells per gene", y = "MSE of log2(x_true)", fill = "n_guides",
         title = sprintf("fit_cis accuracy -- guide_shape = %s", shape)) +
    theme_bw()
  save_plot(p, sprintf("4_cis_mse_%s", shape), height = 12)
}